In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

In [4]:
data = pd.read_csv('credit_risk_dataset.csv')

In [5]:
data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Data Train-Test Splitting

In [6]:
response_variable = 'loan_status'

y = data[response_variable]
x = data.drop(columns=[response_variable], axis=1)

print(x.shape)
print(y.shape)

(32581, 11)
(32581,)


In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, stratify=y, test_size=0.3, random_state=42
)

# Validate splitting
print('X train shape :', x_train.shape)
print('y train shape :', y_train.shape)
print('X test shape  :', x_test.shape)
print('y test shape  :', y_test.shape)

X train shape : (22806, 11)
y train shape : (22806,)
X test shape  : (9775, 11)
y test shape  : (9775,)


In [8]:
y_train.value_counts()

loan_status
0    17831
1     4975
Name: count, dtype: int64

In [9]:
y_test.value_counts()

loan_status
0    7642
1    2133
Name: count, dtype: int64

In [10]:
data_train = pd.concat((x_train, y_train), axis=1)
data_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
11491,26,62000,RENT,1.0,DEBTCONSOLIDATION,B,10000,11.26,0.16,N,2,0
3890,23,39000,MORTGAGE,3.0,EDUCATION,C,5000,12.98,0.13,N,4,0
17344,24,35000,RENT,1.0,DEBTCONSOLIDATION,A,12000,6.54,0.34,N,2,1
13023,24,86000,RENT,1.0,HOMEIMPROVEMENT,B,12000,10.65,0.14,N,3,0
29565,42,38400,RENT,4.0,MEDICAL,B,13000,NaN,0.34,N,11,1


# Question 1

In [11]:
default = data_train[response_variable]
loan_intent = data_train['loan_intent']

loan_intent.head()

11491    DEBTCONSOLIDATION
3890             EDUCATION
17344    DEBTCONSOLIDATION
13023      HOMEIMPROVEMENT
29565              MEDICAL
Name: loan_intent, dtype: object

In [12]:
loan_intent.value_counts()

loan_intent
EDUCATION            4520
MEDICAL              4237
VENTURE              4007
PERSONAL             3856
DEBTCONSOLIDATION    3668
HOMEIMPROVEMENT      2518
Name: count, dtype: int64

### Model Fitting

In [13]:
import statsmodels.api as sm

loan_intent_default = pd.get_dummies(loan_intent).astype(int)

loan_intent_sm = sm.add_constant(loan_intent_default)
logit_model = sm.Logit(endog=default, exog=loan_intent_sm)
resul_loan_intent = logit_model.fit()

print(resul_loan_intent.summary())

Optimization terminated successfully.
         Current function value: 0.517104
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22799
Method:                           MLE   Df Model:                            6
Date:                Mon, 16 Jun 2025   Pseudo R-squ.:                 0.01419
Time:                        22:38:45   Log-Likelihood:                -11793.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                 2.667e-70
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.0966   2.76e+05  -3.98e-06      1.000    -5.4e+05     5.4e+05
DEBTCONS

#### Thus we have the logit model :
$$
\begin{align*}
\text{logit(loan-intent)} &=\beta_{0} + \beta_{1}k_{1} + \beta_{2}k_{2} + \beta_{3}k_{3} + \beta_{4}k_{4} + \beta_{5}k_{5} + \beta_{6}k_{6} \\
\\
&=\beta_{0} + \beta_{1}(\text{DEBTCONSOLIDATION}) + \beta_{2}(\text{EDUCATION}) + \beta_{3}(\text{HOMEIMPROVEMENT}) + \beta_{4}(\text{PERSONAL}) + \beta_{5}(\text{MEDICAL}) + \beta_{6}(\text{VENTURE}) \\
\\
\text{logit(home)} &= -1.0966 + 0.1440(\text{DEBTCONSOLIDATION}) - 0.4617(\text{EDUCATION}) + 0.0472(\text{HOMEIMPROVEMENT}) + 0.0984(\text{MEDICAL}) - 0.2884(\text{PERSONAL}) - 0.6361(\text{VENTURE})   \\
\end{align*}
$$

In [14]:
# Print the parameter estimate of bk for k=0,1,2,3,4,5
bk_loan_intent = resul_loan_intent.params
bk_loan_intent

const               -1.096615
DEBTCONSOLIDATION    0.143971
EDUCATION           -0.461663
HOMEIMPROVEMENT      0.047176
MEDICAL              0.098409
PERSONAL            -0.288383
VENTURE             -0.636127
dtype: float64

### Interpretation
---

Interpretation for $\beta_{0}$.
- $\beta_{0}$ is the logit value when $k_{1} = k_{2} = k_{3} = k_{4} = k_{5} = k_{6} = 0 $ or $x =$ reference.
- Thus, $\beta_{0}$ is the logit or log odds of success of reference category.
- $\text{odds(reference)}=\exp(\beta_{0})$

In [15]:
# Calculate the odds of OTHER, OWN, RENT compared with MORTGAGE
odds_ratio_loan_intent = np.exp(bk_loan_intent[0:7])

print(f"1. OR (DEBTCONSOLIDATION) = {odds_ratio_loan_intent['DEBTCONSOLIDATION']:.2f}")
print(f"2. OR (EDUCATION) = {odds_ratio_loan_intent['EDUCATION']:.2f}")
print(f"3. OR (HOMEIMPROVEMENT) = {odds_ratio_loan_intent['HOMEIMPROVEMENT']:.2f}")
print(f"4. OR (MEDICAL) = {odds_ratio_loan_intent['MEDICAL']:.2f}")
print(f"5. OR (PERSONAL) = {odds_ratio_loan_intent['PERSONAL']:.2f}")
print(f"6. OR (VENTURE) = {odds_ratio_loan_intent['VENTURE']:.2f}")

1. OR (DEBTCONSOLIDATION) = 1.15
2. OR (EDUCATION) = 0.63
3. OR (HOMEIMPROVEMENT) = 1.05
4. OR (MEDICAL) = 1.10
5. OR (PERSONAL) = 0.75
6. OR (VENTURE) = 0.53


##### Prove of 'default' by 'loan_intent'

In [16]:
# Proportion of 'default' by 'home'
crosstab_loan_intent = pd.crosstab(loan_intent,
                                   default,
                                   margins = True)
crosstab_loan_intent

loan_status,0,1,All
loan_intent,,,
DEBTCONSOLIDATION,2647,1021,3668
EDUCATION,3734,786,4520
HOMEIMPROVEMENT,1865,653,2518
MEDICAL,3096,1141,4237
PERSONAL,3084,772,3856
VENTURE,3405,602,4007
All,17831,4975,22806


In [17]:
# Calculate odds of each category
crosstab_loan_intent['odds'] = crosstab_loan_intent[1] / crosstab_loan_intent[0]

crosstab_loan_intent

loan_status,0,1,All,odds
loan_intent,,,,
DEBTCONSOLIDATION,2647,1021,3668,0.385720
EDUCATION,3734,786,4520,0.210498
HOMEIMPROVEMENT,1865,653,2518,0.350134
MEDICAL,3096,1141,4237,0.368540
PERSONAL,3084,772,3856,0.250324
VENTURE,3405,602,4007,0.176799
All,17831,4975,22806,0.279008


In [18]:
# Extract the odds of each category
odds_debt = crosstab_loan_intent['odds']['DEBTCONSOLIDATION']
odds_edu = crosstab_loan_intent['odds']['EDUCATION']
odds_home_imp = crosstab_loan_intent['odds']['HOMEIMPROVEMENT']
odds_medical = crosstab_loan_intent['odds']['MEDICAL']
odds_personal = crosstab_loan_intent['odds']['PERSONAL']
odds_venture = crosstab_loan_intent['odds']['VENTURE']

print(f"The odds of default for DEBTCONSOLIDATION is {odds_debt:.2f}.")
print(f"The odds of default for EDUCATION is {odds_edu:.2f}.")
print(f"The odds of default for HOMEIMPROVEMENT is {odds_home_imp:.2f}.")
print(f"The odds of default for MEDICAL is {odds_medical:.2f}.")
print(f"The odds of default for PERSONAL is {odds_personal:.2f}.")
print(f"The odds of default for VENTURE is {odds_venture:.2f}.")

The odds of default for DEBTCONSOLIDATION is 0.39.
The odds of default for EDUCATION is 0.21.
The odds of default for HOMEIMPROVEMENT is 0.35.
The odds of default for MEDICAL is 0.37.
The odds of default for PERSONAL is 0.25.
The odds of default for VENTURE is 0.18.


In [19]:
# Modelling with sklearn

# Load Statistics package
from sklearn.linear_model import LogisticRegression

# Create the object
model_loan_intent_sk = LogisticRegression(penalty=None)

# Use dummy variable 'history_default' as predictor
model_loan_intent_sk.fit(X=loan_intent_default, y=default)

LogisticRegression(penalty=None)

# Print the parameter estimate of b1
b1_loan_intent = model_loan_intent_sk.coef_
b1_loan_intent

In [20]:
# Calculate the OR between history_default=1 and history_default=0
odds_loan_intent = np.exp(model_loan_intent_sk.coef_)

print(f"OR (ever default, never default) = {odds_loan_intent[0][0]:.2f}")

OR (ever default, never default) = 1.16


### Interpretation

> Debtors who have been in default tend to default again than those who have never been in default. The odds of default for debtors who have been in default is 1.16 times the odds for those who have never been in default.

## Question 2

### Loan_percent_income
---

In [73]:
loan_percent_income = data_train['loan_percent_income']

loan_percent_income_sm = sm.add_constant(loan_percent_income)
model_loan_percent_income = sm.Logit(endog=default, exog=loan_percent_income_sm)
result_loan_percent_income = model_loan_percent_income.fit()

print(result_loan_percent_income.summary())

Optimization terminated successfully.
         Current function value: 0.457157
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22804
Method:                           MLE   Df Model:                            1
Date:                Mon, 16 Jun 2025   Pseudo R-squ.:                  0.1285
Time:                        23:09:14   Log-Likelihood:                -10426.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -2.8775      0.038    -75.615      0.000      -2.952      -2.803
lo

#### Thus we have the logit model :
$$
\begin{align*}
\text{logit(loan-intent)} &=\beta_{0} + \beta_{1}k_{1} \\
\\
&=\beta_{0} + \beta_{1}(\text{loan-percent-income}) \\
\\
\text{logit(home)} &= -2.8775 + 8.3345(\text{loan-percent-income}) \\
\end{align*}
$$

In [103]:
# Print the parameter estimate of bk for k=0,1,2,3,4,5
bk_loan_percent_income = result_loan_percent_income.params
bk_loan_percent_income

const                 -2.877484
loan_percent_income    8.334473
dtype: float64

In [106]:
# Modelling with sklearn

# Load Statistics package
from sklearn.linear_model import LogisticRegression

# Create the object
model_loan_percent_income_sk = LogisticRegression(penalty=None)

# Use dummy variable 'history_default' as predictor
model_loan_percent_income_sk.fit(X=loan_percent_income_sm, y=default)

LogisticRegression(penalty=None)

In [107]:
# Calculate the OR between history_default=1 and history_default=0
odds_loan_percent_income = np.exp(model_loan_percent_income_sk.coef_)

print(f"OR (ever default, never default) = {odds_loan_percent_income[0][0]:.2f}")

OR (ever default, never default) = 0.24


Debtors who have been in default tend to default again than those who have never been in default. The odds of default for debtors who have been in default is 0.24 times the odds for those who have never been in default.

### Loan_int_rate
---

In [67]:
from sklearn.impute import SimpleImputer


loan_int_rate = data_train[['loan_int_rate']]
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
loan_int_rate = imp_median.fit(loan_int_rate).transform(loan_int_rate)

loan_int_rate = pd.DataFrame(loan_int_rate, columns=['loan_int_rate'])

loan_int_rate_sm = sm.add_constant(loan_int_rate)
model_loan_int_rate = sm.Logit(endog=list(default), exog=loan_int_rate_sm)
result_loan_int_rate = model_loan_int_rate.fit()

print(result_loan_int_rate.summary())

Optimization terminated successfully.
         Current function value: 0.474757
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22804
Method:                           MLE   Df Model:                            1
Date:                Mon, 16 Jun 2025   Pseudo R-squ.:                 0.09492
Time:                        23:05:32   Log-Likelihood:                -10827.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -4.3449      0.074    -58.384      0.000      -4.491      -4.199
loan_int_rate     0.

#### Thus we have the logit model :
$$
\begin{align*}
\text{logit(loan-intent)} &=\beta_{0} + \beta_{1}k_{1} \\
\\
&=\beta_{0} + \beta_{1}(\text{loan-int-rate}) \\
\\
\text{logit(home)} &= -4.3449 + 0.2634(\text{loan-int-rate}) \\
\end{align*}
$$

In [75]:
# Print the parameter estimate of bk for k=0,1,2,3,4,5
bk_loan_int_rate = result_loan_int_rate.params
bk_loan_int_rate

const           -4.344865
loan_int_rate    0.263366
dtype: float64

In [78]:
# Modelling with sklearn

# Load Statistics package
from sklearn.linear_model import LogisticRegression

# Create the object
model_loan_int_rate_sk = LogisticRegression(penalty=None)

# Use dummy variable 'history_default' as predictor
model_loan_int_rate_sk.fit(X=loan_int_rate, y=default)

LogisticRegression(penalty=None)

In [84]:
# Calculate the OR between history_default=1 and history_default=0
odds_loan_int_rate = np.exp(model_loan_int_rate_sk.coef_)

print(f"OR (ever default, never default) = {odds_loan_int_rate[0][0]:.2f}")

OR (ever default, never default) = 1.30


Debtors who have been in default tend to default again than those who have never been in default. The odds of default for debtors who have been in default is 1.30 times the odds for those who have never been in default.

### Cb_person_cred_hist_length
---

In [85]:
import statsmodels.api as sm

credit_history = data_train['cb_person_cred_hist_length']

credit_history_sm = sm.add_constant(credit_history)
credit_history_model = sm.Logit(endog=default, exog=credit_history_sm)
result_credit_history = credit_history_model.fit()

print(result_credit_history.summary())

Optimization terminated successfully.
         Current function value: 0.524488
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22804
Method:                           MLE   Df Model:                            1
Date:                Mon, 16 Jun 2025   Pseudo R-squ.:               0.0001168
Time:                        23:27:59   Log-Likelihood:                -11961.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                   0.09463
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -1.2381      0.028    -44.190      0.000      

#### Thus we have the logit model :
$$
\begin{align*}
\text{logit(cb-person-cred-hist-length)} &=\beta_{0} + \beta_{1}k_{1} \\
\\
&=\beta_{0} + \beta_{1}(\text{cb-person-cred-hist-length}) \\
\\
\text{logit(cb-person-cred-hist-length)} &= -1.2381 - 0.0067(\text{cb-person-cred-hist-length}) \\
\end{align*}
$$

In [86]:
# Print the parameter estimate of bk for k=0,1,2,3,4,5
bk_credit_history = result_credit_history.params
bk_credit_history

const                        -1.238076
cb_person_cred_hist_length   -0.006665
dtype: float64

In [93]:
# Modelling with sklearn

# Load Statistics package
from sklearn.linear_model import LogisticRegression

# Create the object
model_loan_int_rate_sk = LogisticRegression(penalty=None)

# Use dummy variable 'history_default' as predictor
model_loan_int_rate_sk.fit(X=credit_history_sm, y=default)

LogisticRegression(penalty=None)

In [89]:
# Calculate the OR between history_default=1 and history_default=0
odds_loan_int_rate = np.exp(model_loan_int_rate_sk.coef_)

print(f"OR (ever default, never default) = {odds_loan_int_rate[0][0]:.2f}")

OR (ever default, never default) = 0.54


Debtors who have been in default tend to default again than those who have never been in default. The odds of default for debtors who have been in default is 0.54 times the odds for those who have never been in default.

## Question 3

In [98]:
loan_intent_dummies = pd.get_dummies(data_train['loan_intent'], drop_first=True)
loan_intent_dm = sm.add_constant(loan_intent_dummies.astype(int))
model_loan_intent = sm.Logit(endog=y_train, exog=loan_intent_dm)
result_loan_intent = model_loan_intent.fit()

print(result_loan_intent.summary())

Optimization terminated successfully.
         Current function value: 0.517104
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22800
Method:                           MLE   Df Model:                            5
Date:                Sun, 15 Jun 2025   Pseudo R-squ.:                 0.01419
Time:                        22:14:57   Log-Likelihood:                -11793.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                 3.071e-71
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.9526      0.037    -25.859      0.000      -1.025      -0.880
EDUCATION     

In [99]:
loan_intent_dummies = pd.get_dummies(data_train['loan_int_rate'], drop_first=True)
loan_intent_dm = sm.add_constant(loan_intent_dummies.astype(int))
model_loan_intent = sm.Logit(endog=y_train, exog=loan_intent_dm)
result_loan_intent = model_loan_intent.fit()

print(result_loan_intent.summary())

/Users/rudi/Documents/pacmann/credit-scoring-analytics/venv/lib/python3.9/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/Users/rudi/Documents/pacmann/credit-scoring-analytics/venv/lib/python3.9/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


         Current function value: inf
         Iterations: 35


LinAlgError: Singular matrix